# Previsão do preço do Bitcoin com LSTM — o estudo em fases

**Objetivo:** descobrir o que **realmente** melhora a previsão quando mexemos no modelo, separando efeito de ruído.
O notebook segue a ordem das descobertas: **cada fase é uma tentativa de melhorar o modelo**, motivada pelo que a fase
anterior revelou. Dentro de cada fase, a metodologia é a mesma (ver `METODOLOGIA.md`): blocos sequenciais em que cada um
herda automaticamente o campeão do anterior, ruído entre seeds como régua, e o teste revelado só no fim da fase.

## Roteiro

- **Fase 1 — Ajustar os hiperparâmetros da lista (BTC 2017–2023)** — `lstm_b0_referencia`, `lstm_b1_entrada`, `lstm_b2_capacidade`, `lstm_b3_ativacao_init`, `lstm_b4_otimizacao`, `lstm_b4_checagem`, `lstm_b5_regularizacao`, `lstm_b6_complementar_capacidade`, `lstm_b7_complementar_epocas`, `lstm_b8_bonus_celula`, `lstm_b9_bonus_augmentation`, `lstm_b10_ablacao`
  - *Descoberta:* Na validação, a cadeia de blocos levou o Theil de 0,989 (LSTM padrão) para 0,979: ~2% melhor que o passeio aleatório. Os ganhos vieram da janela de 20 dias e da ativação ReLU na célula; capacidade, regularização, épocas e GRU não mudaram nada além do ruído.
- **Fase 2 — Revisão: configurações divergentes distorciam a análise?** — sem treino (reanálise)
  - *Descoberta:* Nenhuma decisão da fase 1 mudou: os 12 campeões são os mesmos, nenhuma configuração divergente chegou a uma final e a herança entre blocos é a mesma (divergências já ficavam no fim do ranking pelos erros enormes).
- **Fase 3 — Mais dados: série BTC-USD de 2014 a 2026** — `f3_referencia`, `f3_transplante`
  - *Descoberta:* Com mais dados, o LSTM padrão bateu o passeio aleatório no teste nas 5 seeds (Theil 0,983–0,995; seed 42: 0,988), ~1% melhor e acima do ruído. O ganho está no tamanho do movimento, não na direção: o acerto de subida/queda segue em ~50%.
- **Fase 4 — Informação de fora do preço e atenção** — `f4_contexto`, `f4_atencao`
  - *Descoberta:* Nenhuma fonte de contexto melhorou a validação: calendário, derivativos (funding) e sentimento empataram com só retornos dentro do ruído; on-chain e todas juntas pioraram além do ruído (Theil 0,997 e 0,996 contra 0,990). No teste, todas ficaram piores que só retornos (0,993 a 1,057 contra 0,988): mais informação levou a mais sobreajuste.
- **Fase 5 — O espaço de busca completo (série longa)** — `f5_referencia_modelos`, `f5_b1_entrada`, `f5_b2_normalizacao`, `f5_b3_arquitetura`, `f5_b4_ativacao_init`, `f5_b5_perda`, `f5_b6_otimizacao`, `f5_checagem`, `f5_b7_regularizacao`, `f5_b8_epocas`, `f5_bonus_augmentation`, `f5_ablacao`
  - *Descoberta:* Ainda não executada: esta fase é a próxima a rodar (os números aparecem nas células de análise).
- **Fase 6 — Lacunas da fase 5: ativações, dropout recorrente, horizonte e TimeGAN** — `f6_b1_ativacao_init`, `f6_b2_regularizacao`, `f6_horizonte`, `f6_bonus_timegan`
  - *Descoberta:* Ainda não executada.
- **Fase 7 — Mais dados reais: várias criptomoedas no treino** — `f7_referencia`, `f7_ativos`, `f7_bonus_timegan`
  - *Descoberta:* Ainda não executada.
- **Fase 8 — Mais dados reais: dados por hora** — `f8_referencia`, `f8_janela`
  - *Descoberta:* Ainda não executada.

As descobertas acima são das execuções de referência (fase 1 no Kaggle com 2× T4; fases 3 e 4 numa GPU local). Ao rodar,
o notebook recalcula tudo, e as células de análise de cada fase mostram os números desta execução.

## Protocolo (vale para todas as fases)

- **Validação walk-forward em 5 folds:** o fold k treina em tudo antes do bloco de validação k e valida nele. As partições são por data e são as mesmas em todos os experimentos da fase (comparações pareadas); um embargo de h dias impede que alvos do treino entrem na validação.
- **Métrica de decisão:** `val/rmse` (menor é melhor). Também são registradas `mse`, `mae`, `mape` (% no preço), `theil` (erro ÷ erro do passeio aleatório; < 1 = bate a referência), **`pocid`** (% de acerto na direção: D_t = 1 se (P_t − P_t−1)(P̂_t − P̂_t−1) > 0), `da` (acerto de direção em relação ao preço atual), `skill`, `r2` e `ic`.
- **Triagem e confirmação:** cada configuração roda os folds [3, 4, 5]; as 3 melhores completam os 5 folds, e o campeão é a melhor média nos 5 folds (nunca a média parcial da triagem).
- **Divergência:** um fold diverge se `val/theil` > 10 ou se a métrica não é finita. Configurações divergentes não chegam a finais, não viram campeãs e ficam fora das médias (listadas à parte).
- **Ruído entre seeds:** o LSTM padrão roda com 5 seeds na referência de cada série; um efeito só é "real" se passar de 2× o desvio da diferença pareada entre seeds.
- **Teste:** usado só no fim de cada fase, para os campeões e as referências.

**Resultados em disco** (W&B e GitHub são só espelhos): cada série tem sua pasta, `outputs/` (série 2017–2023) e
`outputs_btc_longo/` (série 2014–2026). Em cada uma: `{exp_name}/` por experimento (parâmetros, histórico por época,
resultados por fold, pesos, previsões), `_grids/{bloco}/` por bloco (configurações, rankings, campeão, logs) e
`_relatorio/` com tabelas e figuras.

**Tempo estimado:** a fase 1 tem ~1.000 treinos de fold (~1–3 h no Kaggle com 2× T4); as fases 3 e 4 somam ~100 treinos na
série longa (~20–40 min); a fase 5 tem ~1.200 treinos na série longa (~2–4 h); as fases 6 a 8 somam ~600 treinos, parte
deles com dezenas de milhares de janelas (várias moedas, dados por hora) e com o TimeGAN (~2–4 h). Para rodar só uma parte,
use `FASES` (e `RESUME_FROM` para trazer os resultados das fases que já rodaram). Se a sessão cair ou passar de 12 h, retome (veja abaixo): o que terminou é pulado.

## Como executar

O notebook é o mesmo em qualquer ambiente: clona o código do GitHub (ou usa a cópia local), acha os dados onde estiverem
e pula o que não estiver disponível. **Nada além de Python + GPU é obrigatório** (W&B e GitHub são opcionais).

### No Kaggle (execução principal)

1. **Dados:** os CSVs vêm no repositório clonado (série curta na raiz, série longa e séries externas em `data/`). Anexar o
   CSV curto como Dataset é opcional: o notebook o acha em `/kaggle/input` pelo nome ou pelas colunas `date`/`close`.
2. **Notebook:** *Code → New Notebook → File → Import Notebook* → `Kaggle_LSTM.ipynb`.
3. **Secrets:** *Add-ons → Secrets* → `GITHUB_TOKEN` e `WANDB_API_KEY`, marcados como anexados.
4. **Settings:** *Accelerator* **GPU T4 ×2**, *Internet* **On** (clone do código, pip, W&B e envio dos resultados).
5. *Save Version → **Save & Run All (Commit)***: roda em segundo plano (limite de 12 h); a saída fica em *Output*.
6. Para validar antes em minutos: `MODO_TESTE = True` (3 configurações por bloco, 2 épocas, pastas `*_teste`).

### No Colab ou Jupyter local

- **Colab:** *Runtime → Change runtime type → GPU*; o código é clonado; tokens em *Colab Secrets*; saída em `/content`.
- **Jupyter local:** abra o notebook dentro do repositório; tokens no `.env` (modelo em `.env.example`); saída no repositório.

**W&B.** Uma run por configuração, agrupada por bloco, num projeto por série (`if702-miniproject-2-lstm` e
`if702-miniproject-2-lstm-longo`); uma run de resumo por bloco (hiperparâmetro × métrica) e uma por fase. O teste só é
enviado na run de resumo da fase, depois de revelado.

**GitHub.** Com `GITHUB_TOKEN` com permissão de escrita (*Contents: Read and write*), ao fim de cada bloco os resultados vão
para o branch `resultados`, em `RUN_NAME/<pasta>/`.

**Retomada.** Para continuar uma execução interrompida: o mesmo `RUN_NAME` com `RESUME_FROM = "github"`, ou
`RESUME_FROM = "/kaggle/input/<output anterior>"` (a pasta que contém `outputs/` e `outputs_btc_longo/`). Para reaproveitar
uma fase já rodada (ex.: a fase 1 do Kaggle), basta que a pasta dela esteja lá: nada que já terminou é treinado de novo.

# Este notebook roda só: Fase 6 — Lacunas da fase 5: ativações, dropout recorrente, horizonte e TimeGAN

As fases anteriores já rodaram: anexe o Output (ou um Dataset) com as pastas de resultados delas (`outputs/`,
`outputs_btc_longo/`, …) e aponte `RESUME_FROM` para a pasta que as contém. Se a sessão chegar a `HORAS_LIMITE`, o
notebook para de começar treinos, salva e termina: rode de novo com `RESUME_FROM` apontando para o Output desta versão,
e ele continua de onde parou (nada que terminou é treinado de novo).

# 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
import os

REPO_URL = "https://github.com/diegoflyra/if702-miniproject-2.git"  # repositório com src/, grids/, config/
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfão só de resultados
RUN_NAME = ""                    # vazio = "lstm-AAAAMMDD-HHMM"; fixe um nome para retomar pelo GitHub
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
FASES = ["fase6"]  # fases a executar (as demais só são lidas, se já tiverem resultados)
WORKERS_PER_GPU = 2              # experimentos simultâneos por GPU (LSTMs pequenos; o Kaggle tem 4 CPUs)
CPU_WORKERS = 1                  # sem GPU
DATA_PATH = ""                   # CSV (ou pasta) da série curta; vazio = raiz do repo → /kaggle/input → /content
SYNC_PESOS = False               # enviar também os .pth ao GitHub
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "")
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida (3 configs por bloco, 2 épocas) em pastas *_teste, para validar o
                                 # notebook inteiro em minutos antes da execução completa
HORAS_LIMITE = 10.5              # depois disso não começa treinos novos e encerra limpo (o Kaggle corta em 12 h);
                                 # para continuar, rode de novo com RESUME_FROM apontando para o Output desta versão
BACKUP_MIN = 30                  # atualiza os .zip de resultados a cada N minutos durante os blocos

In [ ]:
import os
import shutil
import subprocess
import sys
import time


def _load_dotenv(path=".env"):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


_load_dotenv()


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(path):
    return all(os.path.exists(os.path.join(path, p)) for p in ("src/grid_search.py", "grids", "config/estudo.json"))


GITHUB_TOKEN = _secret("GITHUB_TOKEN")
here = os.getcwd()
if _is_repo(here):
    REPO_DIR = here
    print(f"Código: cópia local em {REPO_DIR}")
else:
    REPO_DIR = "/tmp/lstm-acoes"
    clone_url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    if _is_repo(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
    else:
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        cloned = subprocess.run(["git", "clone", "-q", clone_url, REPO_DIR])
        if cloned.returncode != 0 or not _is_repo(REPO_DIR):
            raise RuntimeError("Falha ao clonar o repositório. Repositório privado exige GITHUB_TOKEN "
                               "(Kaggle Secrets, Colab Secrets ou variável de ambiente), ou abra o notebook "
                               "a partir de uma cópia local do projeto (pasta com src/, grids/ e config/).")
    print(f"Código: {REPO_DIR}")

os.chdir(REPO_DIR)
%cd {REPO_DIR}
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

In [ ]:
import importlib.util

skip, pkgs = [], []
with open("requirements.txt", encoding="utf-8") as f:
    for line in f:
        pkg = line.strip()
        if not pkg or pkg.startswith("#"):
            continue
        name = pkg.split("==")[0].split(">=")[0].split("<=")[0].split("~=")[0].split("[")[0].strip().lower()
        if name == "torch" and importlib.util.find_spec(name) is not None:
            skip.append(name)
            continue
        pkgs.append(pkg)
if skip:
    print("Já instalado, não reinstalar (preserva CUDA do ambiente):", ", ".join(skip))
if pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

In [ ]:
import json

import pandas as pd

if os.path.isdir("/kaggle/working"):
    WORKDIR = "/kaggle/working"
elif os.path.isdir("/content"):
    WORKDIR = "/content"
else:
    WORKDIR = REPO_DIR
os.environ["EXP_OUTPUT_ROOT"] = WORKDIR
os.environ["EXP_OUTPUT_SUFFIX"] = "_teste" if MODO_TESTE else ""  # o teste nunca se mistura com a execução real
os.environ.pop("EXP_OUTPUT_DIR", None)
GRID_EXTRA = "--max_configs 3 --epochs 2" if MODO_TESTE else ""
GRID_EXTRA_B0 = "--epochs 2" if MODO_TESTE else ""  # a referência roda completa: as 5 seeds medem o ruído
if MODO_TESTE:
    print("MODO_TESTE: 3 configurações por bloco, 2 épocas, pastas *_teste (não servem como resultado).")

RUN_NAME = RUN_NAME or time.strftime("lstm-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
os.environ.update({"RUN_NAME": RUN_NAME, "RESULTS_REPO_URL": RESULTS_REPO_URL, "RESULTS_BRANCH": RESULTS_BRANCH,
                   "GITHUB_TOKEN": GITHUB_TOKEN, "WANDB_ENTITY": WANDB_ENTITY})
if DATA_PATH:
    os.environ["DATA_PATH"] = DATA_PATH
print(f"RUN_NAME = {RUN_NAME} | resultados em {WORKDIR}")

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
import common
import data as data_mod
import github_sync
import report_utils as rep
import wandb_report

if RESUME_FROM == "github":
    github_sync.pull(WORKDIR)
elif RESUME_FROM:
    subdirs = [d for d in os.listdir(RESUME_FROM) if d.startswith("outputs") and os.path.isdir(os.path.join(RESUME_FROM, d))]
    for d in subdirs or [os.path.basename(os.path.normpath(RESUME_FROM))]:
        src = os.path.join(RESUME_FROM, d) if subdirs else RESUME_FROM
        shutil.copytree(src, os.path.join(WORKDIR, d), dirs_exist_ok=True)
    print(f"Resultados anteriores copiados de {RESUME_FROM}; o que já foi concluído será pulado.")

wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key:
    os.environ["WANDB_API_KEY"] = wandb_key
    os.environ.pop("WANDB_MODE", None)
    print("W&B: chave carregada.")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print(f"W&B desativado ({'USAR_WANDB = False' if not USAR_WANDB else 'sem chave'}).")
if not (GITHUB_TOKEN and RESULTS_REPO_URL):
    print("GitHub: sem GITHUB_TOKEN/RESULTS_REPO_URL, os resultados ficam só em disco (e nos .zip).")


def usar_estudo(config):
    """Troca a série/estudo ativo: dados, partições, pasta de resultados e projeto do W&B."""
    global OUTPUTS
    os.environ["ESTUDO_CONFIG"] = config
    est = common.load_study()
    os.environ["WANDB_PROJECT"] = est.get("wandb_project", "if702-miniproject-2-lstm")
    OUTPUTS = common.output_dir()
    print(f"Estudo ativo: {est.get('titulo')} ({config}) → {OUTPUTS} | W&B: {os.environ['WANDB_PROJECT']}")


os.environ["PRAZO_SESSAO"] = str(time.time() + HORAS_LIMITE * 3600)
os.environ["BACKUP_MIN"] = str(BACKUP_MIN)
_PARADA = os.path.join(WORKDIR, "_PARADO_POR_TEMPO")
if os.path.isfile(_PARADA):
    os.remove(_PARADA)
print(f"Prazo: novos treinos até {time.strftime('%H:%M', time.localtime(time.time() + HORAS_LIMITE * 3600))} "
      f"({HORAS_LIMITE} h); backup a cada {BACKUP_MIN} min.")


def continuar():
    """False depois que o prazo da sessão foi atingido: as células seguintes se pulam e o notebook termina limpo."""
    if os.path.isfile(_PARADA) or time.time() > float(os.environ["PRAZO_SESSAO"]):
        open(_PARADA, "w").close()
        print("Prazo da sessão atingido: célula pulada. Para continuar, rode de novo com RESUME_FROM apontando para o "
              "Output desta versão (o que terminou não é treinado de novo).")
        return False
    return True


def executar(fase):
    """True se a fase está em FASES; senão as células de treino dela são puladas (e as análises leem o que houver)."""
    if fase not in FASES:
        print(f"{fase} fora de FASES: treino pulado; as análises abaixo usam os resultados já existentes, se houver.")
    return fase in FASES


def backup(bloco=""):
    """Backup parcial ao fim de cada bloco: <pasta>.zip (sem .pth) + espelho no GitHub, se configurado."""
    import zipfile

    zpath = OUTPUTS + ".zip"
    with zipfile.ZipFile(zpath + ".tmp", "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(OUTPUTS):
            for name in files:
                if not name.endswith((".pth", ".tmp")):
                    full = os.path.join(root, name)
                    z.write(full, os.path.relpath(full, WORKDIR))
    os.replace(zpath + ".tmp", zpath)
    print(f"{os.path.basename(zpath)}: {os.path.getsize(zpath) / 1e6:.1f} MB")
    github_sync.push(OUTPUTS, weights=SYNC_PESOS, message=f"{RUN_NAME}: {bloco or 'backup'}")

In [ ]:
# Verificações rápidas antes de gastar GPU (sem treino): GPUs, modelos, métricas, augmentation e todos os grids
if shutil.which("nvidia-smi"):
    !nvidia-smi -L
else:
    print("nvidia-smi não encontrado; o treino usa o device que o PyTorch enxergar (CPU se não houver GPU).")
!python tests/check_models.py
!python tests/check_metrics.py
!python tests/check_augment.py
!python tests/check_grids.py

# Fase 6 — Lacunas da fase 5: ativações, dropout recorrente, horizonte e TimeGAN

**Motivação.** Na fase 5, o campeão de arquitetura saiu bidirecional e a célula LSTM própria não suportava bidirecional: 72 combinações de ativações e todas as 160 com dropout recorrente foram descartadas. A célula própria agora é bidirecional, e os dois blocos são refeitos a partir do campeão da fase 5. A fase também mede o efeito do horizonte de previsão (1, 5 e 20 dias) e, como bônus, dados sintéticos com TimeGAN, com diagnósticos para saber se o gerador aprende o BTC ou só o ruído.

**Blocos desta fase:**

- `f6_b1_ativacao_init` — Fase 6 · Ativações e inicialização (refeito): Quais ativações (na célula LSTM e nas densas) e qual esquema de inicialização funcionam melhor com a estrutura campeã? (lstm_activation × activation × weight_init (aleatória, 40))
- `f6_b2_regularizacao` — Fase 6 · Regularização com dropout recorrente (refeito): Quanto e onde aplicar dropout, e quanta penalização e corte de gradiente? (dropout × rnn_dropout × input_dropout × recurrent_dropout × weight_decay × grad_clip (aleatória, 60))
- `f6_horizonte` — Fase 6 · Horizonte de previsão: Prever o preço de 5 ou 20 dias à frente é mais fácil que prever o de amanhã? (horizon)
- `f6_bonus_timegan` — Fase 6 · Bônus — Dados sintéticos com TimeGAN (série longa): Janelas sintéticas geradas por um TimeGAN ajudam o LSTM, ou o gerador só reproduz o ruído? (gan)

**Série:** `data/btc-usd_yahoo_2014-09-17_2026-09-23.csv`, de 2014-09-17 a 2026-09-23; 5 folds de validação de 365 dias; teste a partir de 2025-09-24.

In [ ]:
usar_estudo("config/estudo_btc_longo.json")
manifesto = data_mod.prepare_prices()
display(data_mod.describe_folds())
!python tests/check_data.py
!python tests/check_features.py

## Fase 6 · Ativações e inicialização (refeito)

**Pergunta:** Quais ativações (na célula LSTM e nas densas) e qual esquema de inicialização funcionam melhor com a estrutura campeã?

Refaz o bloco de ativações da fase 5, que perdeu 72 das combinações: o campeão de arquitetura é bidirecional e a célula LSTM própria (usada para ativações ≠ tanh) não suportava bidirecional. Agora suporta (a direção reversa reproduz o nn.LSTM bidirecional com diferença de 1e-7). Itens 4 e 6 da lista. **Célula LSTM:** as portas são sempre sigmoid; varia a ativação da candidata e da saída (tanh é o padrão; sigmoid, softsign e ReLU usam uma célula própria, mais lenta que o cuDNN). **Densas:** ReLU, tanh, sigmoid e ELU; se a campeã não tiver camada densa, esse eixo não tem efeito e as combinações equivalentes são descartadas. A saída é sempre linear, porque o alvo é contínuo (sigmoid e softmax na saída são para classificação). **Inicialização:** pesos pequenos e aleatórios; `padrao` = PyTorch U(±1/√h), `uniforme` = U(±0,05), `normal` = N(0; 0,05), `xavier` (Glorot), `glorot_ortogonal` (padrão do Keras: ortogonal na recorrência e bias de esquecimento 1) e `he` (Kaiming, pensado para ReLU). Espaço de 4 × 4 × 6 = 96 combinações, com busca aleatória de 40.

**Base:** o melhor campeão entre `f5_b8_epocas`.

| Eixo | Valores |
|---|---|
| `lstm_activation` | `tanh`, `sigmoid`, `softsign`, `relu` |
| `activation` | `relu`, `tanh`, `sigmoid`, `elu` |
| `weight_init` | `padrao`, `uniforme`, `normal`, `xavier`, `glorot_ortogonal`, `he` |

**96 combinações possíveis; busca aleatória de 40** (seed 0). A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** Se a ativação da célula muda algo além de tanh (ReLU pode explodir sem limite), e se a inicialização afeta a convergência (`melhor_epoca_media`) e a variância entre folds.

In [ ]:
if continuar():
    display(rep.show_champion("f5_b8_epocas"))

In [ ]:
if continuar() and executar("fase6"):
    !python src/grid_search.py grids/f6_b1_ativacao_init.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `f6_b1_ativacao_init`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
if continuar():
    display(rep.grid_ranking("f6_b1_ativacao_init", "triagem"))

In [ ]:
if continuar():
    display(rep.discarded_configs("f6_b1_ativacao_init"))  # combinações descartadas antes do treino e o motivo

In [ ]:
if continuar():
    display(rep.diverged_configs("f6_b1_ativacao_init"))  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f6_b1_ativacao_init`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
if continuar():
    importancia, efeitos = rep.axis_importance("f6_b1_ativacao_init")
    display(importancia)
    display(efeitos)

In [ ]:
if continuar():
    display(rep.heatmap("f6_b1_ativacao_init", row="weight_init", col="lstm_activation"))

In [ ]:
if continuar():
    display(rep.heatmap("f6_b1_ativacao_init", row="weight_init", col="lstm_activation", value="melhor_epoca_media"))

In [ ]:
if continuar():
    display(rep.heatmap("f6_b1_ativacao_init", row="activation", col="lstm_activation"))

In [ ]:
if continuar():
    display(rep.param_effect("f6_b1_ativacao_init"))

In [ ]:
if continuar():
    display(rep.plot_grid_bars("f6_b1_ativacao_init"))

#### Confirmação das finalistas e campeão — `f6_b1_ativacao_init`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
if continuar():
    display(rep.grid_ranking("f6_b1_ativacao_init", "final"))

In [ ]:
if continuar():
    rep.show_champion("f6_b1_ativacao_init")
    display(rep.plot_finalists("f6_b1_ativacao_init"))

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
if continuar():
    display(rep.paired_comparison(rep.base_config_name("f6_b1_ativacao_init"), "f6_b1_ativacao_init__*"))

In [ ]:
if continuar():
    display(rep.paired_comparison(rep.base_config_name("f6_b1_ativacao_init"), "f6_b1_ativacao_init__*", metric="val/pocid"))

### 📝 Análise — Fase 6 · Ativações e inicialização (refeito)

- **Ativação da célula LSTM:** _…_
- **Ativação das densas:** _…_
- **Inicialização: afetou a convergência ou só o ruído entre folds?** _…_
- **Ganho sobre o Bloco 2 (validação):** _…_

In [ ]:
backup("f6_b1_ativacao_init")

## Fase 6 · Regularização com dropout recorrente (refeito)

**Pergunta:** Quanto e onde aplicar dropout, e quanta penalização e corte de gradiente?

Refaz o bloco de regularização da fase 5, em que todas as 160 combinações com dropout recorrente foram descartadas pelo mesmo motivo (célula própria sem suporte a bidirecional). Todas as formas de dropout em LSTM: depois da recorrência e entre as densas (`dropout`), entre camadas empilhadas (`rnn_dropout`), na entrada (`input_dropout`) e **recorrente** (`recurrent_dropout`: no estado oculto entre um passo de tempo e o seguinte, com a mesma máscara em toda a sequência, como no Keras). Mais o decaimento dos pesos (L2) e o corte da norma do gradiente. Espaço de 5 × 3 × 3 × 4 × 4 × 3 = 2.160 combinações, com busca aleatória de 60; mais épocas e paciência.

**Base:** o melhor campeão entre `f6_b1_ativacao_init`.

| Eixo | Valores |
|---|---|
| `dropout` | `0`, `0.1`, `0.2`, `0.3`, `0.5` |
| `rnn_dropout` | `0`, `0.2`, `0.4` |
| `input_dropout` | `0`, `0.1`, `0.2` |
| `recurrent_dropout` | `0`, `0.1`, `0.2`, `0.3` |
| `weight_decay` | `0`, `1e-05`, `0.0001`, `0.001` |
| `grad_clip` | `0.5`, `1`, `5` |

**2160 combinações possíveis; busca aleatória de 60** (seed 0). Fixos no bloco: `epochs=200`, `patience=20`. A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** O gap treino–validação, e se o dropout recorrente (o que faltava) ajuda.

In [ ]:
if continuar():
    display(rep.show_champion("f6_b1_ativacao_init"))

In [ ]:
if continuar() and executar("fase6"):
    !python src/grid_search.py grids/f6_b2_regularizacao.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `f6_b2_regularizacao`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
if continuar():
    display(rep.grid_ranking("f6_b2_regularizacao", "triagem"))

In [ ]:
if continuar():
    display(rep.discarded_configs("f6_b2_regularizacao"))  # combinações descartadas antes do treino e o motivo

In [ ]:
if continuar():
    display(rep.diverged_configs("f6_b2_regularizacao"))  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f6_b2_regularizacao`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
if continuar():
    importancia, efeitos = rep.axis_importance("f6_b2_regularizacao")
    display(importancia)
    display(efeitos)

In [ ]:
if continuar():
    display(rep.heatmap("f6_b2_regularizacao", row="dropout", col="recurrent_dropout"))

In [ ]:
if continuar():
    display(rep.heatmap("f6_b2_regularizacao", row="dropout", col="recurrent_dropout", value="gap/rmse_mean"))

In [ ]:
if continuar():
    display(rep.param_effect("f6_b2_regularizacao"))

In [ ]:
if continuar():
    display(rep.plot_grid_bars("f6_b2_regularizacao"))

#### Confirmação das finalistas e campeão — `f6_b2_regularizacao`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
if continuar():
    display(rep.grid_ranking("f6_b2_regularizacao", "final"))

In [ ]:
if continuar():
    rep.show_champion("f6_b2_regularizacao")
    display(rep.plot_finalists("f6_b2_regularizacao"))

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
if continuar():
    display(rep.paired_comparison(rep.base_config_name("f6_b2_regularizacao"), "f6_b2_regularizacao__*"))

In [ ]:
if continuar():
    display(rep.paired_comparison(rep.base_config_name("f6_b2_regularizacao"), "f6_b2_regularizacao__*", metric="val/pocid"))

### 📝 Análise — Fase 6 · Regularização com dropout recorrente (refeito)

- **Qual dropout ajudou (se algum)?** _…_
- **Dropout recorrente:** _…_
- **Weight decay e corte de gradiente:** _…_
- **Gap e subajuste:** _…_

In [ ]:
backup("f6_b2_regularizacao")

## Fase 6 · Horizonte de previsão

**Pergunta:** Prever o preço de 5 ou 20 dias à frente é mais fácil que prever o de amanhã?

O campeão de regularização com horizonte de 1, 5 e 20 dias: o alvo passa a ser o log-retorno de h dias (o preço de t+h), e a referência é o passeio aleatório no mesmo horizonte (prever que o preço de t+h é o de t). O embargo entre treino e validação cresce junto (h dias). O RMSE de horizontes diferentes não é comparável (retornos maiores, erros maiores), por isso este bloco decide pelo **Theil**, que já é relativo ao passeio aleatório do mesmo horizonte. Com h > 1, dias vizinhos compartilham parte do período previsto, então a POCID e a acurácia direcional ficam correlacionadas entre dias. **Regra da célula lenta:** se o campeão da regularização usar dropout recorrente (ou ativação ≠ tanh), ~10× mais lento, e não for melhor que a melhor configuração rápida do bloco além do ruído entre seeds, este e os blocos seguintes herdam a rápida (registrado em `base.json`).

**Base:** o melhor campeão entre `f6_b2_regularizacao`.

| Eixo | Valores |
|---|---|
| `horizon` | `1`, `5`, `20` |

**3 combinações.** A base é re-treinada sem mudanças (referência pareada). Todas as configurações rodam os 5 folds.

**O que observar:** Se o Theil cai (e a POCID sobe) com horizontes maiores, onde há menos ruído relativo.

In [ ]:
if continuar():
    display(rep.show_champion("f6_b2_regularizacao"))

In [ ]:
if continuar() and executar("fase6"):
    !python src/grid_search.py grids/f6_horizonte.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

In [ ]:
if continuar():
    display(rep.discarded_configs("f6_horizonte"))  # combinações descartadas antes do treino e o motivo

In [ ]:
if continuar():
    display(rep.diverged_configs("f6_horizonte"))  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f6_horizonte`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
if continuar():
    importancia, efeitos = rep.axis_importance("f6_horizonte", metric="val/theil")
    display(importancia)
    display(efeitos)

In [ ]:
if continuar():
    display(rep.plot_grid_bars("f6_horizonte"))

#### Confirmação das finalistas e campeão — `f6_horizonte`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
if continuar():
    display(rep.grid_ranking("f6_horizonte", "final"))

In [ ]:
if continuar():
    rep.show_champion("f6_horizonte")
    display(rep.plot_finalists("f6_horizonte"))

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
if continuar():
    display(rep.paired_comparison(rep.base_config_name("f6_horizonte"), "f6_horizonte__*", metric="val/theil"))

In [ ]:
if continuar():
    display(rep.paired_comparison(rep.base_config_name("f6_horizonte"), "f6_horizonte__*", metric="val/pocid"))

### 📝 Análise — Fase 6 · Horizonte de previsão

- **Com qual horizonte o modelo mais se afasta do passeio aleatório?** _…_
- **A direção (POCID, acurácia direcional) melhora em horizontes maiores?** _…_

In [ ]:
backup("f6_horizonte")

## Fase 6 · Bônus — Dados sintéticos com TimeGAN (série longa)

**Pergunta:** Janelas sintéticas geradas por um TimeGAN ajudam o LSTM, ou o gerador só reproduz o ruído?

TimeGAN (Yoon et al., 2019) treinado dentro de cada fold, só com as janelas de treino (a janela e o alvo juntos, como uma sequência), com 1.000 ou 3.000 iterações por etapa. **Mistura:** a cada batch de treino, janelas sintéticas são somadas às reais na proporção indicada. **Só sintético (TSTR, train on synthetic, test on real):** o LSTM treina só com o sintético e é avaliado na validação real; se ficar muito pior que treinar com dados reais, o gerador aprendeu ruído, não o BTC. **Diagnósticos por fold:** um classificador tenta separar janelas reais de sintéticas (0 = indistinguíveis), e os fatos estilizados do retorno (curtose, autocorrelação do retorno e do |retorno|) são comparados entre real e sintético. O risco apontado no desenho do experimento é que, numa série tão ruidosa e movida por eventos externos, o gerador reproduza o ruído e não as propriedades do BTC; os diagnósticos medem isso diretamente.

**Base:** o melhor campeão entre `f6_b2_regularizacao`.

| Eixo | Valores |
|---|---|
| `gan` | **sem TimeGAN** (`timegan=none`), **mistura 0,5×** (`timegan=mistura`, `timegan_ratio=0.5`, `timegan_iter=1000`), **mistura 1×** (`timegan=mistura`, `timegan_ratio=1`, `timegan_iter=1000`), **mistura 2×** (`timegan=mistura`, `timegan_ratio=2`, `timegan_iter=1000`), **mistura 1× (3000 it)** (`timegan=mistura`, `timegan_ratio=1`, `timegan_iter=3000`), **só sintético (TSTR)** (`timegan=so_sintetico`, `timegan_iter=1000`), **só sintético (TSTR, 3000 it)** (`timegan=so_sintetico`, `timegan_iter=3000`) |

**7 combinações.** A base é re-treinada sem mudanças (referência pareada). Todas as configurações rodam os 5 folds.

**O que observar:** Os diagnósticos do gerador (fatos estilizados, discriminativo), o TSTR e o Δ pareado da mistura sobre o treino só com dados reais.

In [ ]:
if continuar():
    display(rep.show_champion("f6_b2_regularizacao"))

In [ ]:
if continuar() and executar("fase6"):
    !python src/grid_search.py grids/f6_bonus_timegan.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

In [ ]:
if continuar():
    display(rep.discarded_configs("f6_bonus_timegan"))  # combinações descartadas antes do treino e o motivo

In [ ]:
if continuar():
    display(rep.diverged_configs("f6_bonus_timegan"))  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f6_bonus_timegan`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
if continuar():
    importancia, efeitos = rep.axis_importance("f6_bonus_timegan")
    display(importancia)
    display(efeitos)

In [ ]:
if continuar():
    display(rep.plot_grid_bars("f6_bonus_timegan"))

#### Confirmação das finalistas e campeão — `f6_bonus_timegan`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
if continuar():
    display(rep.grid_ranking("f6_bonus_timegan", "final"))

In [ ]:
if continuar():
    rep.show_champion("f6_bonus_timegan")
    display(rep.plot_finalists("f6_bonus_timegan"))

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
if continuar():
    display(rep.paired_comparison(rep.base_config_name("f6_bonus_timegan"), "f6_bonus_timegan__*"))

In [ ]:
if continuar():
    display(rep.paired_comparison(rep.base_config_name("f6_bonus_timegan"), "f6_bonus_timegan__*", metric="val/pocid"))

#### Diagnósticos do TimeGAN

O gerador reproduz as propriedades do BTC ou só o ruído? Fatos estilizados do retorno (real × sintético), score discriminativo e o TSTR (linha "só sintético").

In [ ]:
if continuar():
    display(rep.timegan_diagnostics("f6_bonus_timegan"))

### 📝 Análise — Fase 6 · Bônus

- **O gerador reproduz as caudas pesadas e os aglomerados de volatilidade do BTC?** _…_
- **TSTR: o LSTM treinado só com sintético prevê o BTC real?** _…_
- **A mistura melhorou a validação além do ruído?** _…_
- **Mais iterações do TimeGAN melhoraram o gerador?** _…_

In [ ]:
backup("f6_bonus_timegan")

## Fase 6 · O que realmente melhorou o modelo (validação)

A **cadeia de decisões** reconstrói, pela herança entre blocos, o caminho do LSTM padrão até o campeão (`f6_b2_regularizacao`): o que
mudou em cada passo, o Δ pareado por fold, em quantos folds a mudança venceu e se o ganho passa do ruído entre seeds.

In [ ]:
if continuar():
    cadeia = rep.decision_chain("f6_b2_regularizacao")
    display(cadeia)

## Fase 6 · Teste revelado

Até aqui todas as escolhas desta fase usaram só a validação. Agora o período de teste é usado **uma única vez**, para as
referências e o campeão de cada bloco (média ± desvio entre os modelos dos K folds). A concordância entre validação e
teste é a evidência de que não houve vazamento; uma discrepância grande (ex.: mudança de regime) é um achado.

In [ ]:
if continuar():
    final = rep.final_report(["f6_b1_ativacao_init", "f6_b2_regularizacao", "f6_horizonte", "f6_bonus_timegan"])
    display(final)

In [ ]:
if continuar():
    campeao_fase = rep.champion_name("f6_b2_regularizacao")
    display(rep.plot_predictions(campeao_fase))

### O que descobrimos nesta fase

Na execução de referência:

- Ainda não executada.

**📝 Nesta execução:** _…_ (confira nas tabelas acima se os números se repetem)

# Comparação entre as fases

Para cada fase com treino: o passeio aleatório, o LSTM padrão e o campeão da fase, na validação e no teste. É o resumo de
tudo o que as tentativas de melhorar o modelo conseguiram.

In [ ]:
if continuar():
    comparacao = rep.compare_phases()
    display(comparacao)

# Próximos passos

- Mudar o alvo: prever a volatilidade futura (sabidamente mais previsível que o retorno) ou a direção (classificação sobe/lateral/desce). Pede métricas próprias, porque Theil e POCID foram feitos para retorno.
- Dados sintéticos com TimeGAN, treinado dentro de cada fold só com dados de treino. Expectativa baixa: o gerador aprende com os mesmos dados, e o augmentation piorou 13 de 14 variações na fase 1; o que ajudou foi dado real a mais.

## 📝 Conclusões

- **Alguma tentativa fez o LSTM bater o passeio aleatório no teste de forma consistente? Qual e por quanto?** _…_
- **O que os hiperparâmetros da lista mudaram de fato (fase 1), e por que não se sustentou no teste?** _…_
- **Mais dados (fase 3) ou mais informação (fase 4): o que pesou mais?** _…_
- **Acerto de direção (POCID): em algum momento saiu de ~50%?** _…_